# PiCar-X: complete working example

**Run the three code cells in order. No edits or installs required.**

1. Run **Setup** and wait for Ready.
2. Run **Working controller**.
3. Run **Simulation**. Expect **COMPLETE: 5/5 skills checkpoints passed**, then an animated replay.

Open Play / Pause if playback is paused. Reset returns to the start. The timeline slider lets you inspect any moment. This is a self-contained guided skills course: camera labels and distance inputs come from a continuous mock world. Signs and objects stay at fixed locations; the pedestrian crosses and a worker pushes a box to the shoulder. The detour follows a guided curve, not vehicle physics. It is separate from the advanced cul-de-sac.

This completed version is for verification first. There are no TODOs or student checkpoint tasks. If you change a setting later, rerun the controller and simulation cells.


## 1. Setup — run once


In [ ]:
# @title 1. Start the pretend car (run once)
class MockPicarx:
    """A pretend sensor reading and a record of requested actions; no real motors."""
    def __init__(self, label="clear", distance=100):
        self.label = label
        self.distance = distance
        self.action = "stop"
        self.steering = 0
        self.history = []
        self.ultrasonic = self

    def scan(self) -> str:
        return self.label

    def read(self) -> float:
        return self.distance

    def forward(self, speed: float) -> None:
        if not 0 <= speed <= 100:
            raise ValueError("Use a speed between 0 and 100.")
        self.action = "forward" if speed > 0 else "stop"
        self.history.append((self.action, speed))

    def stop(self) -> None:
        self.action = "stop"
        self.history.append(("stop", 0))

    def set_dir_servo_angle(self, angle: float) -> None:
        if not -30 <= angle <= 30:
            raise ValueError("Steering must be between -30 and 30 degrees.")
        self.steering = angle


def check_decisions(function):
    cases = [
        ("clear", "forward", "An empty view should allow movement."),
        ("human", "stop", "Yield while a person is visible."),
        ("stop sign", "stop", "Begin stopping when a sign is seen."),
        ("unknown", "stop", "An unfamiliar reading should not mean go."),
    ]
    passed = 0
    for label, expected, hint in cases:
        try:
            actual = function(label)
            ok = actual == expected
            passed += ok
            print(f"{'PASS' if ok else 'TRY AGAIN'} | {label}: {actual!r}")
            if not ok:
                print("  Hint:", hint, "Return", repr(expected))
        except Exception as error:
            print(f"TRY AGAIN | {label}: {type(error).__name__}: {error}")
    print(f"Decision checks: {passed}/{len(cases)}")


def play_scenario(controller, frames, title):
    car = MockPicarx()
    memory = {}
    print("\nSCENARIO:", title)
    for step, (label, distance) in enumerate(frames, start=1):
        car.label, car.distance = label, distance
        car.history = []
        try:
            controller(car, memory)
        except Exception as error:
            print(f"Step {step}: controller error: {error}")
            return False
        personal_object = globals().get("custom_object", "")
        detour = label == personal_object and distance >= 25
        must_stop = (label != "clear" and not detour) or distance < 25
        unsafe_command = any(action == "forward" for action, speed in car.history) if must_stop else False
        correct = car.action == ("stop" if must_stop else "forward") and not unsafe_command
        if correct and not must_stop:
            expected_angle = (-25 if globals().get("avoid_direction") == "left" else 25) if detour else 0
            if car.steering != expected_angle:
                print(f"TRY AGAIN: steering is {car.steering}; expected {expected_angle}.")
                print("Use your chosen side for the custom object; straighten when the view clears.")
                return False
        print(f"Step {step} | camera={label!r}, distance={distance} cm -> {car.action}, steering={car.steering}")
        if not correct:
            print("TRY AGAIN:", "Stop without issuing a forward command for this hazard." if must_stop
                  else "The view is clear and the object is far away. Let the car move.")
            return False
    print("PASS: all decisions matched this scenario.")
    return True

print("Pretend car ready. No camera or motors are connected.")

# Prepare the same full-course replay used in the driving notebook.
from pathlib import Path
import sys
import json

# The course simulator and its pinned SunFounder base class are bundled here.
work_folder = Path.cwd() / "picarx_notebook"
work_folder.mkdir(exist_ok=True)
bundled_sources = json.loads('{"simulator.py": "\\"\\"\\"Deterministic, standard-library-only PiCar-X navigation simulator.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass, field, replace\\nimport json\\nimport math\\nimport random\\nfrom pathlib import Path\\nfrom typing import Callable\\nfrom car_api import CoursePicarx, ScanLabel\\n\\n\\n@dataclass(frozen=True)\\nclass Circle:\\n    x: float\\n    y: float\\n    radius: float\\n\\n\\n@dataclass(frozen=True)\\nclass Checkpoint:\\n    x: float\\n    y: float\\n    radius: float = 38\\n\\n\\n@dataclass(frozen=True)\\nclass SceneObject:\\n    \\"\\"\\"An object that can be recognized by the virtual camera.\\"\\"\\"\\n\\n    kind: str\\n    x: float\\n    y: float\\n    radius: float = 18\\n    crossing_end: tuple[float, float] | None = None\\n\\n\\n@dataclass(frozen=True)\\nclass Scenario:\\n    name: str\\n    width: int\\n    height: int\\n    start: tuple[float, float, float]\\n    goal: Checkpoint\\n    obstacles: tuple[Circle, ...]\\n    checkpoints: tuple[Checkpoint, ...] = ()\\n    route: tuple[tuple[float, float], ...] = ()\\n    scene_objects: tuple[SceneObject, ...] = ()\\n    max_steps: int = 650\\n    sensor_fault_every: int = 0\\n    seed: int = 1\\n    branches: tuple[tuple[tuple[float, float], ...], ...] = ()\\n    turning_bulb: tuple[float, float, float] | None = None\\n    road_half_width: float | None = None\\n\\n    def road_segments(self):\\n        for road in (self.route, *self.branches):\\n            yield from zip(road, road[1:])\\n\\n\\n@dataclass\\nclass RunResult:\\n    scenario: str\\n    reached_goal: bool\\n    checkpoints_reached: int\\n    total_checkpoints: int\\n    collisions: int\\n    boundary_hits: int\\n    invalid_commands: int\\n    sensor_faults_seen: int\\n    steps: int\\n    distance_traveled: float\\n    steering_changes: int\\n    stopped_safely: bool\\n    human_hits: int = 0\\n    stop_sign_violations: int = 0\\n    stop_signs_obeyed: int = 0\\n    controller_errors: list[str] = field(default_factory=list)\\n    replay: list[dict] = field(default_factory=list)\\n\\n\\ndef clamp(value: float, low: float, high: float) -> float:\\n    return max(low, min(high, value))\\n\\n\\ndef distance(a: tuple[float, float], b: tuple[float, float]) -> float:\\n    return math.hypot(a[0] - b[0], a[1] - b[1])\\n\\n\\ndef point_to_segment(point: tuple[float, float], start: tuple[float, float], end: tuple[float, float]) -> float:\\n    \\"\\"\\"Shortest distance between a point and one route-line segment.\\"\\"\\"\\n    dx, dy = end[0] - start[0], end[1] - start[1]\\n    if dx == 0 and dy == 0:\\n        return distance(point, start)\\n    amount = clamp(((point[0] - start[0]) * dx + (point[1] - start[1]) * dy) / (dx * dx + dy * dy), 0, 1)\\n    projection = (start[0] + amount * dx, start[1] + amount * dy)\\n    return distance(point, projection)\\n\\n\\nclass VirtualUltrasonic:\\n    def __init__(self, car: \\"VirtualPicarx\\") -> None:\\n        self.car = car\\n\\n    def read(self) -> float:\\n        \\"\\"\\"Return virtual centimeters to an obstacle or -1 for a sensor fault.\\"\\"\\"\\n        self.car.sensor_reads += 1\\n        fault_every = self.car.scenario.sensor_fault_every\\n        if fault_every and self.car.sensor_reads % fault_every == 0:\\n            self.car.sensor_faults_seen += 1\\n            return -1.0\\n\\n        heading = math.radians(self.car.heading)\\n        for ray_distance in range(2, 201, 2):\\n            x = self.car.x + math.cos(heading) * ray_distance\\n            y = self.car.y + math.sin(heading) * ray_distance\\n            if not (0 <= x <= self.car.scenario.width and 0 <= y <= self.car.scenario.height):\\n                return float(ray_distance)\\n            if any(distance((x, y), (item.x, item.y)) <= item.radius for item in self.car.scenario.obstacles):\\n                return float(ray_distance)\\n        return 200.0\\n\\n\\nclass VirtualPicarx(CoursePicarx):\\n    \\"\\"\\"SunFounder PiCar-X subclass with simulated motors and sensors.\\"\\"\\"\\n\\n    is_simulation = True\\n\\n    def __init__(self, scenario: Scenario) -> None:\\n        self.scenario = scenario\\n        self.x, self.y, self.heading = scenario.start\\n        self.throttle = 0.0\\n        self.steering = 0.0\\n        self.ultrasonic = VirtualUltrasonic(self)\\n        self.invalid_commands = 0\\n        self.sensor_reads = 0\\n        self.sensor_faults_seen = 0\\n        self.last_scan = \\"clear\\"\\n        self.scene_objects = scenario.scene_objects\\n        self.crossing_people: set[int] = set()\\n\\n    def _number(self, value: object, default: float = 0.0) -> float:\\n        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):\\n            self.invalid_commands += 1\\n            return default\\n        return float(value)\\n\\n    def forward(self, speed: float) -> None:\\n        speed = self._number(speed)\\n        if not 0 <= speed <= 100:\\n            self.invalid_commands += 1\\n        self.throttle = clamp(speed, 0, 100)\\n\\n    def backward(self, speed: float) -> None:\\n        speed = self._number(speed)\\n        if not 0 <= speed <= 100:\\n            self.invalid_commands += 1\\n        self.throttle = -clamp(speed, 0, 100)\\n\\n    def stop(self) -> None:\\n        self.throttle = 0.0\\n\\n    def set_dir_servo_angle(self, angle: float) -> None:\\n        angle = self._number(angle)\\n        if not self.DIR_MIN <= angle <= self.DIR_MAX:\\n            self.invalid_commands += 1\\n        self.steering = clamp(angle, self.DIR_MIN, self.DIR_MAX)\\n\\n    def set_cam_pan_angle(self, angle: float) -> None:\\n        self._number(angle)\\n\\n    def set_cam_tilt_angle(self, angle: float) -> None:\\n        self._number(angle)\\n\\n    def get_grayscale_data(self) -> list[int]:\\n        \\"\\"\\"Return left/center/right ground brightness readings.\\n\\n        A low value means the sensor is over the dark route line, matching the\\n        physical lesson\'s left/center/right grayscale list.\\n        \\"\\"\\"\\n        readings: list[int] = []\\n        heading = math.radians(self.heading)\\n        for side_offset in (-18, 0, 18):\\n            sensor_x = self.x + math.cos(heading) * 16 + math.cos(heading + math.pi / 2) * side_offset\\n            sensor_y = self.y + math.sin(heading) * 16 + math.sin(heading + math.pi / 2) * side_offset\\n            route_distance = min(\\n                point_to_segment((sensor_x, sensor_y), start, end)\\n                for start, end in self.scenario.road_segments()\\n            )\\n            readings.append(250 if route_distance <= 11 else 1200)\\n        return readings\\n\\n    def scan(self) -> ScanLabel:\\n        \\"\\"\\"Return the closest camera-recognized object or ``clear``.\\n\\n        This intentionally hides computer-vision math from beginners. In a\\n        physical build, a camera model would supply the same small vocabulary.\\n        \\"\\"\\"\\n        heading = math.radians(self.heading)\\n        visible: list[tuple[float, str]] = []\\n        for index, item in enumerate(self.scene_objects):\\n            if index in self.crossing_people:\\n                visible.append((0, \\"human\\"))\\n                continue\\n            if item.crossing_end is not None:\\n                continue\\n            dx, dy = item.x - self.x, item.y - self.y\\n            forward = dx * math.cos(heading) + dy * math.sin(heading)\\n            sideways = -dx * math.sin(heading) + dy * math.cos(heading)\\n            # A stop sign is reported only after the car enters the legal\\n            # stopping zone. This keeps recognition and scoring synchronized.\\n            recognition_range = 68 if item.kind == \\"stop sign\\" else 145\\n            if 0 < forward <= recognition_range and abs(math.degrees(math.atan2(sideways, forward))) <= 38:\\n                visible.append((math.hypot(dx, dy), item.kind))\\n        self.last_scan = min(visible)[1] if visible else \\"clear\\"\\n        return self.last_scan\\n\\n    def step(self, seconds: float = 0.12) -> tuple[float, float, float]:\\n        previous = (self.x, self.y)\\n        speed = self.throttle * 0.72\\n        turn_rate = speed * math.tan(math.radians(self.steering)) / 38\\n        self.heading = (self.heading + math.degrees(turn_rate * seconds)) % 360\\n        heading = math.radians(self.heading)\\n        self.x += math.cos(heading) * speed * seconds\\n        self.y += math.sin(heading) * speed * seconds\\n        return previous[0], previous[1], distance(previous, (self.x, self.y))\\n\\n\\ndef scenario_catalog() -> dict[str, Scenario]:\\n    practice = Scenario(\\n        \\"practice\\", 900, 520, (65, 260, 29), Checkpoint(835, 260, 42),\\n        (Circle(350, 235, 38), Circle(610, 220, 32)),\\n        (Checkpoint(230, 350), Checkpoint(490, 350), Checkpoint(720, 225)),\\n        ((65, 260), (230, 350), (490, 350), (720, 225), (835, 260)),\\n        (SceneObject(\\"stop sign\\", 159, 352),\\n         SceneObject(\\"human\\", 687, 307, 12, (639, 219))), max_steps=700,\\n    )\\n    slalom = Scenario(\\n        \\"slalom\\", 900, 520, (65, 260, 0), Checkpoint(835, 260, 42),\\n        (Circle(260, 250, 38), Circle(445, 280, 38), Circle(620, 250, 38), Circle(735, 310, 34)),\\n        (Checkpoint(300, 375), Checkpoint(500, 130), Checkpoint(680, 385)),\\n        ((65, 260), (300, 375), (500, 130), (680, 385), (835, 260)),\\n        (SceneObject(\\"human\\", 275, 355), SceneObject(\\"stop sign\\", 445, 115), SceneObject(\\"human\\", 705, 350)), max_steps=850,\\n    )\\n    tight_turn = Scenario(\\n        \\"tight_turn\\", 760, 620, (70, 520, 270), Checkpoint(680, 90, 42),\\n        (Circle(190, 390, 48), Circle(380, 300, 52), Circle(570, 210, 48)),\\n        (Checkpoint(100, 260), Checkpoint(310, 100), Checkpoint(520, 430)),\\n        ((70, 520), (100, 260), (310, 100), (520, 430), (680, 90)),\\n        (SceneObject(\\"stop sign\\", 55, 275), SceneObject(\\"human\\", 505, 410)), max_steps=900,\\n    )\\n    sensor_fault = Scenario(\\n        \\"sensor_fault\\", 900, 520, (65, 260, 0), Checkpoint(835, 260, 42),\\n        (Circle(350, 250, 38), Circle(600, 315, 38)),\\n        (Checkpoint(250, 150), Checkpoint(520, 150), Checkpoint(730, 250)),\\n        ((65, 260), (250, 150), (520, 150), (730, 250), (835, 260)),\\n        (SceneObject(\\"human\\", 500, 145), SceneObject(\\"stop sign\\", 700, 205)),\\n        max_steps=800, sensor_fault_every=9,\\n    )\\n    rng = random.Random(271828)\\n    unseen_obstacles = tuple(Circle(rng.randint(230, 700), rng.randint(120, 400), rng.randint(34, 55)) for _ in range(4))\\n    unseen = Scenario(\\n        \\"unseen\\", 900, 520, (65, 260, 0), Checkpoint(835, 260, 42), unseen_obstacles,\\n        (Checkpoint(300, 95), Checkpoint(500, 430), Checkpoint(710, 95)),\\n        ((65, 260), (300, 95), (500, 430), (710, 95), (835, 260)),\\n        (SceneObject(\\"stop sign\\", 255, 75), SceneObject(\\"human\\", 520, 405)), max_steps=900, seed=271828,\\n    )\\n    warehouse = replace(\\n        practice, name=\\"warehouse\\", width=1740, height=920, max_steps=2000,\\n        route=(*practice.route, (960, 285), (1130, 365), (1280, 440)),\\n        turning_bulb=(1330, 440, 305),\\n        goal=Checkpoint(915, 302, 42),\\n        checkpoints=(*practice.checkpoints, Checkpoint(980, 285, 52),\\n                     Checkpoint(1115, 315, 54), Checkpoint(1255, 223, 58),\\n                     Checkpoint(1450, 237, 58), Checkpoint(1540, 420, 58),\\n                     Checkpoint(1460, 625, 58), Checkpoint(1240, 640, 58),\\n                     Checkpoint(1115, 490, 54)),\\n        obstacles=(*practice.obstacles, Circle(1015, 315, 27), Circle(1190, 285, 33),\\n                   Circle(1320, 215, 30), Circle(1440, 349, 37),\\n                   Circle(1510, 520, 32), Circle(1345, 620, 35),\\n                   Circle(1205, 535, 30), Circle(1280, 430, 39)),\\n    )\\n    warehouse_alt = replace(\\n        warehouse, name=\\"warehouse_alt\\",\\n        obstacles=(*practice.obstacles, Circle(1015, 262, 27), Circle(1190, 385, 33),\\n                   Circle(1350, 270, 32), Circle(1485, 330, 37),\\n                   Circle(1490, 552, 32), Circle(1335, 575, 39),\\n                   Circle(1160, 510, 30), Circle(1280, 430, 39)),\\n    )\\n    return {item.name: item for item in\\n            (practice, slalom, tight_turn, sensor_fault, unseen, warehouse, warehouse_alt)}\\n\\n\\ndef run_scenario(controller: Callable, scenario: Scenario, *, strict: bool = False) -> RunResult:\\n    car = VirtualPicarx(scenario)\\n    memory: dict = {}\\n    replay: list[dict] = []\\n    collisions = boundary_hits = steering_changes = 0\\n    was_in_collision = was_outside = False\\n    distance_traveled = 0.0\\n    checkpoint_index = 0\\n    previous_steering = car.steering\\n    errors: list[str] = []\\n    reached_goal = False\\n    human_hits = 0\\n    stop_sign_violations = 0\\n    stop_signs_obeyed = 0\\n    human_contacts: set[int] = set()\\n    completed_signs: set[int] = set()\\n    sign_stop_counts: dict[int, int] = {}\\n\\n    crossing_starts: dict[int, int] = {}\\n    stop_lines = {}\\n    for index, item in enumerate(scenario.scene_objects):\\n        if item.kind != \\"stop sign\\" or len(scenario.route) < 2:\\n            continue\\n        start, end = min(scenario.road_segments(),\\n                         key=lambda segment: point_to_segment((item.x, item.y), *segment))\\n        length = distance(start, end)\\n        if length:\\n            dx, dy = (end[0] - start[0]) / length, (end[1] - start[1]) / length\\n            along = clamp((item.x - start[0]) * dx + (item.y - start[1]) * dy, 0, length)\\n            stop_lines[index] = (start[0] + along * dx, start[1] + along * dy, dx, dy)\\n\\n    for step_number in range(scenario.max_steps):\\n        people = list(scenario.scene_objects)\\n        car.crossing_people.clear()\\n        for index, item in enumerate(people):\\n            if item.crossing_end is None:\\n                continue\\n            end_x, end_y = item.crossing_end\\n            midpoint = ((item.x + end_x) / 2, (item.y + end_y) / 2)\\n            if distance((car.x, car.y), midpoint) <= 94:\\n                crossing_starts.setdefault(index, step_number)\\n            if index in crossing_starts:\\n                progress = min(1, (step_number - crossing_starts[index]) / 36)\\n                people[index] = replace(item, x=item.x + (end_x - item.x) * progress,\\n                                        y=item.y + (end_y - item.y) * progress)\\n                # Let the pedestrian approach the lane before the controller yields.\\n                if 0.22 <= progress < 1:\\n                    car.crossing_people.add(index)\\n        car.scene_objects = tuple(people)\\n        try:\\n            controller(car, memory)\\n        except Exception as error:  # Student errors become feedback instead of crashing the evaluator.\\n            errors.append(f\\"Step {step_number}: {type(error).__name__}: {error}\\")\\n            car.stop()\\n            break\\n\\n        if abs(car.steering - previous_steering) >= 8:\\n            steering_changes += 1\\n        previous_steering = car.steering\\n        old_x, old_y, moved = car.step()\\n        distance_traveled += moved\\n\\n        outside = not (12 <= car.x <= scenario.width - 12 and 12 <= car.y <= scenario.height - 12)\\n        hit = any(distance((car.x, car.y), (item.x, item.y)) <= item.radius + 12 for item in scenario.obstacles)\\n        if outside or hit:\\n            if outside and not was_outside:\\n                boundary_hits += 1\\n            if hit and not was_in_collision:\\n                collisions += 1\\n            car.x, car.y = old_x, old_y\\n            car.throttle = 0\\n        was_outside = outside\\n        was_in_collision = hit\\n\\n        for object_index, item in enumerate(car.scene_objects):\\n            object_distance = distance((car.x, car.y), (item.x, item.y))\\n            if item.kind == \\"human\\" and object_distance <= item.radius + 12 and object_index not in human_contacts:\\n                human_hits += 1\\n                human_contacts.add(object_index)\\n                car.x, car.y = old_x, old_y\\n                car.stop()\\n                break\\n            if item.kind == \\"stop sign\\" and object_index not in completed_signs:\\n                # Recognition defines the approach zone. Counting the same\\n                # signal here prevents visual stopping and scoring from drifting.\\n                line = stop_lines.get(object_index)\\n                crossed = False\\n                in_stop_zone = object_distance <= 72\\n                if line:\\n                    line_x, line_y, dx, dy = line\\n                    before = (old_x - line_x) * dx + (old_y - line_y) * dy\\n                    after = (car.x - line_x) * dx + (car.y - line_y) * dy\\n                    lateral = abs((car.y - line_y) * dx - (car.x - line_x) * dy)\\n                    crossed = before < 0 <= after and lateral <= 35\\n                    in_stop_zone = -68 <= after <= 0 and lateral <= 35\\n                if in_stop_zone and moved == 0:\\n                    sign_stop_counts[object_index] = sign_stop_counts.get(object_index, 0) + 1\\n                    if sign_stop_counts[object_index] >= 10:\\n                        stop_signs_obeyed += 1\\n                        completed_signs.add(object_index)\\n                else:\\n                    sign_stop_counts[object_index] = 0\\n                if crossed and object_index not in completed_signs:\\n                    stop_sign_violations += 1\\n                    car.stop()\\n                    break\\n\\n        failure = \\"\\"\\n        if human_hits:\\n            failure = \\"Hit a pedestrian\\"\\n        elif stop_sign_violations:\\n            failure = \\"Ran the stop sign\\"\\n        elif strict and hit:\\n            failure = \\"Hit a box\\"\\n        elif strict and outside:\\n            failure = \\"Crossed the course boundary\\"\\n        elif strict and len(replay) >= 120:\\n            # Allow traffic waits and brief recovery, but end a stuck loop.\\n            earlier = replay[-120]\\n            if (earlier[\\"checkpoint\\"] == checkpoint_index\\n                    and distance((car.x, car.y), (earlier[\\"x\\"], earlier[\\"y\\"])) < 12):\\n                failure = \\"Stalled without making progress\\"\\n        safety_failure = bool(failure)\\n        if safety_failure:\\n            car.stop()\\n        if not safety_failure and checkpoint_index < len(scenario.checkpoints):\\n            point = scenario.checkpoints[checkpoint_index]\\n            if distance((car.x, car.y), (point.x, point.y)) <= point.radius:\\n                checkpoint_index += 1\\n\\n        if not safety_failure and checkpoint_index == len(scenario.checkpoints) and distance((car.x, car.y), (scenario.goal.x, scenario.goal.y)) <= scenario.goal.radius:\\n            reached_goal = True\\n            car.stop()\\n\\n        replay.append({\\n            \\"step\\": step_number, \\"x\\": round(car.x, 2), \\"y\\": round(car.y, 2),\\n            \\"heading\\": round(car.heading, 2), \\"steering\\": round(car.steering, 2),\\n            \\"throttle\\": round(car.throttle, 2), \\"checkpoint\\": checkpoint_index,\\n            \\"scan\\": car.last_scan,\\n            \\"objects\\": [\\n                {**item.__dict__, \\"walking\\": index in crossing_starts\\n                 and step_number < crossing_starts[index] + 36}\\n                for index, item in enumerate(car.scene_objects)\\n            ],\\n            \\"failure\\": failure,\\n        })\\n        if reached_goal or safety_failure:\\n            break\\n\\n    return RunResult(\\n        scenario.name, reached_goal, checkpoint_index, len(scenario.checkpoints), collisions,\\n        boundary_hits, car.invalid_commands, car.sensor_faults_seen, len(replay),\\n        round(distance_traveled, 2), steering_changes, abs(car.throttle) < 0.01,\\n        human_hits, stop_sign_violations, stop_signs_obeyed, errors, replay,\\n    )\\n\\n\\ndef render_replay(result: RunResult, scenario: Scenario, output: Path) -> None:\\n    render_replays([(result, scenario)], output)\\n\\n\\ndef render_replays(\\n    runs: list[tuple[RunResult, Scenario]], output: Path, *,\\n    autoplay: bool = False, evaluation_score: float | None = None,\\n) -> None:\\n    payload = json.dumps([{\\n        \\"result\\": {key: value for key, value in result.__dict__.items() if key != \\"replay\\"},\\n        \\"frames\\": result.replay,\\n        \\"scenario\\": {\\n            \\"name\\": scenario.name, \\"width\\": scenario.width, \\"height\\": scenario.height,\\n            \\"goal\\": scenario.goal.__dict__,\\n            \\"obstacles\\": [item.__dict__ for item in scenario.obstacles],\\n            \\"checkpoints\\": [item.__dict__ for item in scenario.checkpoints],\\n            \\"route\\": scenario.route,\\n            \\"branches\\": scenario.branches,\\n            \\"turningBulb\\": scenario.turning_bulb,\\n            \\"roadHalfWidth\\": scenario.road_half_width,\\n            \\"sceneObjects\\": [item.__dict__ for item in scenario.scene_objects],\\n        },\\n    } for result, scenario in runs]).replace(\\"<\\", \\"\\\\\\\\u003c\\")\\n    template = \\"\\"\\"<!doctype html><html><head><meta charset=\\"utf-8\\"><title>Virtual PiCar-X Replay</title>\\n<style>body{font-family:Arial;background:#071b2e;color:white;margin:0;padding:22px}main{max-width:1100px;margin:auto}.card{background:#102d46;border:1px solid #2d6383;border-radius:14px;padding:16px;margin:12px 0}.views{display:grid;grid-template-columns:2fr 1fr;gap:14px}canvas{width:100%;background:#d9f0df;border-radius:10px}button,input{font-size:16px;margin-right:10px}strong{color:#55d6be}.legend span{margin-right:18px}@media(max-width:800px){.views{grid-template-columns:1fr}}</style></head>\\n<body><main><h1>Virtual PiCar-X Chase View</h1><div class=\\"card\\" id=\\"summary\\"></div><div class=\\"views\\"><canvas id=\\"driver\\" width=\\"760\\" height=\\"470\\"></canvas><canvas id=\\"course\\"></canvas></div>\\n<div class=\\"card\\"><button id=\\"play\\">Play / Pause</button><button id=\\"reset\\">Reset</button><input id=\\"slider\\" type=\\"range\\" min=\\"0\\" value=\\"0\\"><span id=\\"step\\"></span> \\u00b7 <strong id=\\"score\\"></strong></div>\\n<p class=\\"card\\" id=\\"hint\\" role=\\"status\\" hidden></p>\\n__EVALUATION__\\n<div class=\\"legend\\"><span>\\ud83d\\udfe6 Car</span><span>\\ud83d\\udfe5 Obstacle</span><span>\\ud83d\\udfe8 Checkpoint</span><span>\\ud83d\\udfe9 Goal</span></div></main>\\n<script>const runs=__DATA__,c=document.querySelector(\'#course\'),x=c.getContext(\'2d\'),d=document.querySelector(\'#driver\'),v=d.getContext(\'2d\');\\nconst slider=document.querySelector(\'#slider\');\\nconst playButton=document.querySelector(\'#play\');\\nconst selector=document.createElement(\'select\');\\nselector.setAttribute(\'aria-label\',\'Course\');\\nplayButton.before(selector);\\nselector.hidden=runs.length===1;\\nruns.forEach((run,index)=>selector.add(new Option(run.scenario.name.replaceAll(\'_\',\' \'),index)));\\nlet data,s,roads,i=0,t=null;\\nfunction selectCourse(){\\n    pause();\\n    data=runs[Number(selector.value)];\\n    s=data.scenario;\\n    roads=[s.route,...s.branches].filter(route=>route.length>=2).map(routeSamples);\\n    c.width=s.width;\\n    c.height=s.height;\\n    i=0;\\n    slider.value=0;\\n    slider.max=Math.max(0,data.frames.length-1);\\n    playButton.disabled=!data.frames.length;\\n    const result=data.result;\\n    document.querySelector(\'#summary\').textContent=\\n        `${s.name} | Goal: ${result.reached_goal?\'reached\':\'not reached\'} | Checkpoints: ${result.checkpoints_reached}/${result.total_checkpoints} | Collisions: ${result.collisions}`;\\n    document.querySelector(\'#step\').textContent=result.controller_errors[0]??\'\';\\n    showScore(0, !data.frames.length);\\n    showHint(0, !data.frames.length);\\n    d.getContext(\'2d\').clearRect(0,0,d.width,d.height);\\n    draw();\\n}\\nfunction circle(o,color){x.beginPath();x.arc(o.x,o.y,o.radius,0,Math.PI*2);x.fillStyle=color;x.fill();}\\nfunction worldToCamera(o,f){const a=f.heading*Math.PI/180,dx=o.x-f.x,dy=o.y-f.y;return{depth:dx*Math.cos(a)+dy*Math.sin(a),side:-dx*Math.sin(a)+dy*Math.cos(a)}}\\nfunction smoothstep(a,b,value){const t=Math.max(0,Math.min(1,(value-a)/(b-a)));return t*t*(3-2*t)}\\nfunction routeSamples(route){const pts=[];for(let n=0;n<route.length-1;n++){const a=route[n],b=route[n+1],length=Math.hypot(b[0]-a[0],b[1]-a[1]),steps=Math.max(2,Math.ceil(length/18));for(let k=0;k<steps;k++){const t=k/steps;pts.push({x:a[0]+(b[0]-a[0])*t,y:a[1]+(b[1]-a[1])*t})}}const z=route[route.length-1];pts.push({x:z[0],y:z[1]});return pts}\\nfunction perspective(point){\\n    const scale=340/point.depth;\\n    return {x:d.width/2+point.side*scale,y:155+55*scale,depth:point.depth,approach:Math.min(1,70/point.depth)};\\n}\\nfunction projectGround(point,frame){\\n    const camera=worldToCamera(point,frame);\\n    return camera.depth>=12 ? perspective(camera) : null;\\n}\\nfunction roadPolygon(points,frame,color){\\n    const camera=points.map(point=>worldToCamera(point,frame));\\n    const clipped=[];\\n    for(let index=0;index<camera.length;index++){\\n        const a=camera[index],b=camera[(index+1)%camera.length];\\n        if(a.depth>=12)clipped.push(a);\\n        if((a.depth>=12)!==(b.depth>=12)){\\n            const fraction=(12-a.depth)/(b.depth-a.depth);\\n            clipped.push({depth:12,side:a.side+(b.side-a.side)*fraction});\\n        }\\n    }\\n    if(clipped.length<3)return;\\n    v.beginPath();\\n    clipped.map(perspective).forEach((point,index)=>index?v.lineTo(point.x,point.y):v.moveTo(point.x,point.y));\\n    v.closePath();\\n    v.fillStyle=color;\\n    v.fill();\\n}\\nfunction roadEdge(sampledRoute,index,offset){\\n    const point=sampledRoute[index];\\n    const before=sampledRoute[Math.max(0,index-1)],after=sampledRoute[Math.min(index+1,sampledRoute.length-1)];\\n    const dx=after.x-before.x,dy=after.y-before.y,length=Math.hypot(dx,dy)||1;\\n    return {x:point.x-dy/length*offset,y:point.y+dx/length*offset};\\n}\\nfunction drawRoad(frame){\\n    for(const sampledRoute of roads)drawRoadStrip(frame,sampledRoute);\\n    if(s.turningBulb){\\n        const [x,y,radius]=s.turningBulb;\\n        const rim=Array.from({length:64},(_,index)=>{\\n            const angle=index*Math.PI/32;\\n            return {x:x+radius*Math.cos(angle),y:y+radius*Math.sin(angle)};\\n        });\\n        roadPolygon(rim,frame,\'#484b50\');\\n    }\\n}\\nfunction drawRoadStrip(frame,sampledRoute){\\n    function strip(index,left,right,color){\\n        roadPolygon([roadEdge(sampledRoute,index,left),roadEdge(sampledRoute,index,right),\\n                     roadEdge(sampledRoute,index+1,right),roadEdge(sampledRoute,index+1,left)],frame,color);\\n    }\\n    for(let index=0;index<sampledRoute.length-1;index++){\\n        const halfWidth=s.roadHalfWidth ?? (s.turningBulb && sampledRoute[index].x>=835?82:24);\\n        strip(index,-halfWidth-5,halfWidth+5,\'#aab1a6\');\\n        strip(index,-halfWidth,halfWidth,\'#484b50\');\\n        if(halfWidth>24 && s.roadHalfWidth==null)continue;\\n        strip(index,-halfWidth+1,-halfWidth+2,\'#ece8dc\');\\n        strip(index,halfWidth-2,halfWidth-1,\'#ece8dc\');\\n        if(index%3!==2)strip(index,-.8,.8,\'#e8dcaa\');\\n    }\\n}\\nfunction drawCar(width,height){\\n    v.save();\\n    v.translate(width/2,height-70);\\n    v.fillStyle=\'rgba(0,0,0,.35)\';\\n    v.beginPath();\\n    v.ellipse(0,28,72,15,0,0,Math.PI*2);\\n    v.fill();\\n    for(const [left,top,wide,tall] of [[-53,-35,18,30],[35,-35,18,30],[-65,3,23,36],[42,3,23,36]]){\\n        v.fillStyle=\'#111820\';\\n        v.fillRect(left,top,wide,tall);\\n        v.fillStyle=\'#39434d\';\\n        for(let tread=top+4;tread<top+tall;tread+=6)v.fillRect(left+2,tread,wide-4,2);\\n    }\\n    v.fillStyle=\'#273440\';\\n    v.beginPath();\\n    v.moveTo(-45,30);v.lineTo(-34,-33);v.lineTo(34,-33);v.lineTo(45,30);\\n    v.closePath();v.fill();\\n    v.strokeStyle=\'#637381\';v.lineWidth=2;v.stroke();\\n    v.fillStyle=\'#247451\';v.fillRect(-24,-11,48,28);\\n    v.fillStyle=\'#17242e\';v.fillRect(-10,-5,19,14);\\n    v.fillStyle=\'#b7c4ce\';v.fillRect(-22,18,44,8);\\n    v.fillStyle=\'#718392\';v.fillRect(-5,-53,10,34);\\n    v.fillStyle=\'#202a33\';v.fillRect(-14,-62,28,16);\\n    v.fillStyle=\'#10181f\';v.fillRect(-10,-59,20,10);\\n    v.fillStyle=\'#69c3e0\';v.fillRect(-3,-56,6,4);\\n    v.restore();\\n}\\nfunction drawCrosswalk(camera){\\n    if(s.roadHalfWidth!=null){\\n        for(const sign of s.sceneObjects.filter(object=>object.kind===\'stop sign\')){\\n            const stopX=sign.x-25,half=s.roadHalfWidth-4;\\n            roadPolygon([{x:stopX-2,y:260-half},{x:stopX+2,y:260-half},\\n                         {x:stopX+2,y:260+half},{x:stopX-2,y:260+half}],camera,\'#f4efe0\');\\n        }\\n    }\\n    for(const person of s.sceneObjects){\\n        if(!person.crossing_end)continue;\\n        const dx=person.crossing_end[0]-person.x,dy=person.crossing_end[1]-person.y;\\n        const length=Math.hypot(dx,dy);\\n        if(!length)continue;\\n        const alongX=dy/length*5,alongY=-dx/length*5;\\n        for(let offset=.26;offset<.72;offset+=.105){\\n            const x=person.x+dx*offset,y=person.y+dy*offset;\\n            roadPolygon([\\n                {x:x-alongX,y:y-alongY},{x:x+alongX,y:y+alongY},\\n                {x:x+alongX+dx*.06,y:y+alongY+dy*.06},\\n                {x:x-alongX+dx*.06,y:y-alongY+dy*.06}\\n            ],camera,\'#f4efe0\');\\n        }\\n    }\\n}\\nfunction driverView(f){const W=d.width,H=d.height,h=155;let g=v.createLinearGradient(0,0,0,h);g.addColorStop(0,\'#69b9e8\');g.addColorStop(1,\'#d9f4ff\');v.fillStyle=g;v.fillRect(0,0,W,h);g=v.createLinearGradient(0,h,0,H);g.addColorStop(0,\'#65a75f\');g.addColorStop(1,\'#254f2e\');v.fillStyle=g;v.fillRect(0,h,W,H-h);for(let n=0;n<7;n++){const tx=35+n*118;v.fillStyle=\'#386b35\';v.beginPath();v.arc(tx,h-8-(n%2)*12,18+(n%3)*5,0,Math.PI*2);v.fill();v.fillStyle=\'#79553a\';v.fillRect(tx-3,h-2,6,28)}const angle=f.heading*Math.PI/180;const camera={...f,x:f.x-64*Math.cos(angle),y:f.y-64*Math.sin(angle)};drawRoad(camera);drawCrosswalk(camera);const objects=[...s.obstacles.map(o=>({...o,kind:\'obstacle\'})),...f.objects];objects.map(o=>({o,p:projectGround(o,camera)})).filter(q=>q.p&&q.p.depth<1600&&Math.abs(q.p.x-W/2)<W*.65).sort((a,b)=>b.p.depth-a.p.depth).forEach(q=>{const scale=Math.min(2.2,150/q.p.depth),ground=q.p.y;const sx=q.p.x;const stride=q.o.walking?Math.sin(f.step*.48)*8:0;if(q.o.kind===\'human\'){v.strokeStyle=\'#f9c34e\';v.lineWidth=Math.max(3,7*scale);v.beginPath();v.arc(sx,ground-45*scale,9*scale,0,Math.PI*2);v.moveTo(sx,ground-35*scale);v.lineTo(sx,ground-8*scale);v.moveTo(sx,ground-25*scale);v.lineTo(sx-14*scale,ground-15*scale);v.moveTo(sx,ground-25*scale);v.lineTo(sx+14*scale,ground-15*scale);v.moveTo(sx,ground-8*scale);v.lineTo(sx-12*scale,ground+(10+stride)*scale);v.moveTo(sx,ground-8*scale);v.lineTo(sx+12*scale,ground+10*scale);v.stroke()}else if(q.o.kind===\'stop sign\'){v.fillStyle=\'#70757b\';v.fillRect(sx-2*scale,ground-38*scale,4*scale,43*scale);v.fillStyle=\'#c9252d\';v.strokeStyle=\'white\';v.lineWidth=Math.max(1,2*scale);v.beginPath();for(let n=0;n<8;n++){const a=Math.PI/8+n*Math.PI/4,px=sx+18*scale*Math.cos(a),py=ground-51*scale+18*scale*Math.sin(a);n?v.lineTo(px,py):v.moveTo(px,py)}v.closePath();v.fill();v.stroke();v.fillStyle=\'white\';v.font=`bold ${Math.max(7,8*scale)}px Arial`;v.textAlign=\'center\';v.fillText(\'STOP\',sx,ground-48*scale)}else{drawBox(sx,ground,scale*q.o.radius/26)}});drawCar(W,H);v.fillStyle=\'rgba(4,15,27,.82)\';v.fillRect(0,H-36,W,36);v.fillStyle=\'#55d6be\';v.font=\'bold 18px Arial\';v.textAlign=\'left\';v.fillText(`${f.failure || (f.scan===\'human\' && !f.throttle ? \'YIELDING - PEDESTRIAN CROSSING\' : \'\') || \'CAMERA: \'+f.scan.toUpperCase()+\'   SPEED: \'+Math.abs(f.throttle).toFixed(0)}`,16,H-12)}\\nfunction drawBox(x,y,scale){\\n    v.save();v.translate(x,y);v.scale(scale,scale);\\n    v.fillStyle=\'#ad784a\';v.fillRect(-20,-31,40,31);\\n    v.fillStyle=\'#dfaf75\';v.beginPath();\\n    v.moveTo(-20,-31);v.lineTo(-12,-42);v.lineTo(28,-42);v.lineTo(20,-31);v.fill();\\n    v.fillStyle=\'#81522f\';v.beginPath();\\n    v.moveTo(20,0);v.lineTo(28,-11);v.lineTo(28,-42);v.lineTo(20,-31);v.fill();\\n    v.fillStyle=\'#edd0a2\';v.fillRect(-4,-31,8,26);\\n    v.restore();\\n}\\nfunction showScore(checkpoints, finished){\\n    const label=finished?\'Final course score\':\'Course score\';\\n    document.querySelector(\'#score\').textContent=\\n        `${label}: ${checkpoints}/${data.result.total_checkpoints} checkpoint points`;\\n}\\nfunction showHint(checkpoints, finished, failure){\\n    const panel=document.querySelector(\'#hint\');\\n    panel.hidden=!finished || data.result.reached_goal;\\n    if(panel.hidden)return;\\n    const hints={\\n        \'Ran the stop sign\':\'Check what your code does when car.scan() returns \\"stop sign\\". Does it bring the car to a full stop before the line?\',\\n        \'Hit a pedestrian\':\'Check how your code reacts to \\"human\\". Does the car wait until the crossing is clear?\',\\n        \'Hit a box\':\'Check your distance reading just before contact. Did the car start steering around the box early enough?\',\\n        \'Crossed the course boundary\':\'Check the direction and duration of your last turn on the map.\',\\n        \'Stalled without making progress\':\'Scrub back a few steps. Is the car repeating the same turn or staying stopped after the way clears?\'\\n    };\\n    const target=checkpoints<data.result.total_checkpoints\\n        ? `Checkpoint ${checkpoints+1} not reached.`\\n        : \'Finish area not reached.\';\\n    const error=data.result.controller_errors[0];\\n    const hint=error ? `Start by investigating the controller error: ${error}`\\n        : hints[failure] ?? \'Trace the car on the map and compare its path with the next target.\';\\n    const message=`${target} ${hint}`;\\n    if(panel.textContent!==message)panel.textContent=message;\\n}\\nfunction draw(position=i){if(!data.frames.length)return; i=Math.floor(position);x.clearRect(0,0,c.width,c.height);x.fillStyle=\'#d9f0df\';x.fillRect(0,0,c.width,c.height);if(s.turningBulb){const [x,y,radius]=s.turningBulb;circle({x,y,radius},\'#687079\');}x.strokeStyle=\'#253746\';x.lineWidth=8;for(const road of [s.route,...s.branches]){x.beginPath();road.forEach((p,n)=>n?x.lineTo(p[0],p[1]):x.moveTo(p[0],p[1]));x.stroke();if(s.turningBulb){x.strokeStyle=\'#687079\';x.lineWidth=164;x.beginPath();road.filter(p=>p[0]>=835).forEach((p,n)=>n?x.lineTo(p[0],p[1]):x.moveTo(p[0],p[1]));x.stroke();}}s.obstacles.forEach(o=>circle(o,\'#d9534f\'));data.frames[i].objects.forEach(o=>circle(o,o.kind===\'human\'?\'#7a4fa3\':\'#d22\'));s.checkpoints.forEach((p,n)=>{circle(p,n<data.frames[i].checkpoint?\'#8ccf72\':\'#f4c95d\');x.fillStyle=\'#172b3a\';x.fillText(n+1,p.x-3,p.y+4)});circle(s.goal,\'#42b883\');x.strokeStyle=\'#246b95\';x.lineWidth=3;x.beginPath();data.frames.slice(0,i+1).forEach((f,n)=>n?x.lineTo(f.x,f.y):x.moveTo(f.x,f.y));x.stroke();const current=data.frames[i],next=data.frames[Math.min(i+1,data.frames.length-1)],fraction=position-i;const delta=((next.heading-current.heading+540)%360)-180;const f={...current,step:position,objects:current.objects.map((object,index)=>{const following=object.id?next.objects.find(item=>item.id===object.id):next.objects[index];if(!following||following.kind!==object.kind)return {...object};return {...object,x:object.x+(following.x-object.x)*fraction,y:object.y+(following.y-object.y)*fraction}}),x:current.x+(next.x-current.x)*fraction,y:current.y+(next.y-current.y)*fraction,heading:current.heading+delta*fraction};x.save();x.translate(f.x,f.y);x.rotate(f.heading*Math.PI/180);x.fillStyle=\'#247ba0\';x.fillRect(-14,-9,28,18);x.fillStyle=\'white\';x.fillRect(4,-5,8,10);x.restore();driverView(f);slider.value=i;document.querySelector(\'#step\').textContent=`Step ${i+1}/${data.frames.length}`;showScore(current.checkpoint,i===data.frames.length-1);showHint(current.checkpoint,i===data.frames.length-1,current.failure);}\\nfunction pause(){if(t!==null)cancelAnimationFrame(t);t=null;}\\nfunction play(){\\n    if(t!==null){pause();return;}\\n    if(!data.frames.length)return;\\n    if(i===data.frames.length-1)i=0;\\n    let position=i;\\n    let previousTime;\\n    function animate(now){\\n        // Animation timestamps can precede the time recorded in the click handler.\\n        const elapsed=Math.min(50,Math.max(0,now-(previousTime??now)));\\n        previousTime=now;\\n        const crossing=data.frames[Math.floor(position)].objects.some(object=>object.walking);\\n        position=Math.min(data.frames.length-1,position+elapsed/(crossing?110:35));\\n        draw(position);\\n        t=position<data.frames.length-1?requestAnimationFrame(animate):null;\\n    }\\n    t=requestAnimationFrame(animate);\\n}\\nplayButton.onclick=play;\\ndocument.querySelector(\'#reset\').onclick=()=>{pause();i=0;draw()};\\nslider.oninput=e=>{pause();i=Number(e.target.value);draw()};\\nselector.onchange=selectCourse;\\nselectCourse();__AUTOPLAY__</script></body></html>\\"\\"\\"\\n    evaluation = (\\n        f\'<p class=\\"card\\">Automated rubric score: <strong>{evaluation_score:g}/90</strong>\'\\n        \' \\u00b7 <a style=\\"color:#55d6be\\" href=\\"score_report.txt\\">Read the score report</a></p>\'\\n        if evaluation_score is not None else \\"\\"\\n    )\\n    page = template.replace(\\"__EVALUATION__\\", evaluation).replace(\\n        \\"__AUTOPLAY__\\", \\"play();\\" if autoplay else \\"\\"\\n    )\\n    output.write_text(page.replace(\\"__DATA__\\", payload), encoding=\\"utf-8\\")\\n", "run_simulation.py": "\\"\\"\\"Run one visible Virtual PiCar-X practice scenario.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport importlib.util\\nfrom pathlib import Path\\n\\nfrom simulator import render_replays, run_scenario, scenario_catalog\\n\\n\\nHERE = Path(__file__).resolve().parent\\n\\n\\ndef load_controller(path: Path):\\n    spec = importlib.util.spec_from_file_location(\\"student_controller\\", path)\\n    if spec is None or spec.loader is None:\\n        raise RuntimeError(f\\"Could not load {path}\\")\\n    module = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(module)\\n    if not hasattr(module, \\"control_step\\"):\\n        raise AttributeError(\\"student_controller.py must define control_step(car, memory)\\")\\n    return module.control_step\\n\\n\\ndef main(*, strict: bool = False) -> None:\\n    parser = argparse.ArgumentParser(description=\\"Run the Virtual PiCar-X simulator\\")\\n    parser.add_argument(\\"--controller\\", type=Path, default=HERE / \\"student_controller.py\\")\\n    default_replay = \\"strict_replay.html\\" if strict else \\"simulation_replay.html\\"\\n    parser.add_argument(\\"--output\\", type=Path, default=HERE / default_replay)\\n    args = parser.parse_args()\\n\\n    scenarios = scenario_catalog()\\n    scenario = scenarios[\\"warehouse\\"]\\n    runs = [(run_scenario(load_controller(args.controller), scenario, strict=strict), scenario)]\\n    render_replays(runs, args.output)\\n    result, _ = runs[0]\\n\\n    print(f\\"Scenario: {result.scenario}\\")\\n    print(f\\"Goal reached: {result.reached_goal}\\")\\n    print(f\\"Checkpoints: {result.checkpoints_reached}/{result.total_checkpoints}\\")\\n    print(f\\"Collisions: {result.collisions}; boundary hits: {result.boundary_hits}\\")\\n    print(f\\"Replay: {args.output}\\")\\n    if result.replay and result.replay[-1][\\"failure\\"]:\\n        print(f\\"Attempt ended: {result.replay[-1][\'failure\']}\\")\\n    if result.controller_errors:\\n        print(\\"Controller error:\\", result.controller_errors[0])\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "evaluate.py": "\\"\\"\\"Transparent scoring harness for the Virtual PiCar-X capstone.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport ast\\nfrom pathlib import Path\\n\\nfrom run_simulation import load_controller\\nfrom simulator import RunResult, run_scenario, scenario_catalog\\n\\n\\nHERE = Path(__file__).resolve().parent\\nCONTROLLER = HERE / \\"student_controller.py\\"\\n\\n\\ndef code_quality(path: Path) -> tuple[float, list[str]]:\\n    source = path.read_text(encoding=\\"utf-8\\")\\n    tree = ast.parse(source)\\n    functions = [node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]\\n    notes: list[str] = []\\n    points = 0.0\\n    if any(node.name == \\"control_step\\" for node in functions):\\n        points += 2\\n    entry = next((node for node in tree.body\\n                  if isinstance(node, ast.FunctionDef) and node.name == \\"control_step\\"), None)\\n    helper_names = {node.name for node in functions} - {\\"control_step\\"}\\n    if entry and any(\\n        isinstance(node, ast.Call) and isinstance(node.func, ast.Name)\\n        and node.func.id in helper_names for node in ast.walk(entry)\\n    ):\\n        points += 2\\n    else:\\n        notes.append(\\"Move a distinct driving decision into a helper function and call it from control_step.\\")\\n    if all((node.end_lineno or node.lineno) - node.lineno <= 35 for node in functions):\\n        points += 2\\n    else:\\n        notes.append(\\"Break long functions into smaller jobs.\\")\\n    names = [node.id for node in ast.walk(tree) if isinstance(node, ast.Name)]\\n    short_names = [name for name in names if len(name) == 1 and name not in {\\"x\\", \\"y\\"}]\\n    if len(short_names) <= 2:\\n        points += 2\\n    else:\\n        notes.append(\\"Use descriptive variable names.\\")\\n    if \\"#\\" in source and any(ast.get_docstring(node) for node in functions):\\n        points += 2\\n    else:\\n        notes.append(\\"Document the reason for important decisions.\\")\\n    return points, notes\\n\\n\\ndef score_runs(results: list[RunResult], quality_points: float) -> tuple[float, list[str]]:\\n    count = len(results)\\n    goal_rate = sum(result.reached_goal for result in results) / count\\n    checkpoint_total = sum(result.total_checkpoints for result in results)\\n    checkpoint_rate = sum(result.checkpoints_reached for result in results) / max(1, checkpoint_total)\\n    crashes = sum(result.collisions + result.boundary_hits for result in results)\\n    human_hits = sum(result.human_hits for result in results)\\n    stop_violations = sum(result.stop_sign_violations for result in results)\\n    invalid = sum(result.invalid_commands for result in results)\\n    errors = sum(bool(result.controller_errors) for result in results)\\n    safely_stopped = sum(\\n        result.stopped_safely and not (result.human_hits or result.stop_sign_violations)\\n        for result in results\\n    ) / count\\n\\n    mission = 25 * goal_rate + 25 * checkpoint_rate\\n    safety = max(0, 15 * checkpoint_rate - crashes * 2.5 - human_hits * 8\\n                 - stop_violations * 5 - invalid * 0.5 - errors * 5)\\n    robustness = 10 * checkpoint_rate * safely_stopped * (1 - errors / count)\\n    efficient_runs = sum(\\n        result.reached_goal and result.steps < 650 and result.steering_changes < 120\\n        for result in results\\n    )\\n    efficiency = 5 * efficient_runs / count\\n    total = round(mission + safety + robustness + efficiency + quality_points, 1)\\n\\n    feedback: list[str] = []\\n    if goal_rate < 0.6:\\n        feedback.append(\\"Mission: tune navigation so the controller reaches more goals.\\")\\n    if crashes:\\n        feedback.append(\\"Safety: slow down sooner or improve escape steering near obstacles and walls.\\")\\n    if human_hits:\\n        feedback.append(\\"Human safety: stop earlier and wait before navigating around a person.\\")\\n    if stop_violations:\\n        feedback.append(\\"Traffic rules: remain stopped long enough before passing a stop sign.\\")\\n    if goal_rate and efficiency < 5:\\n        feedback.append(\\"Efficiency: reduce unnecessary steering changes or long recovery loops.\\")\\n    return min(90.0, total), feedback\\n\\n\\ndef write_score_report(results: list[RunResult], controller_path: Path, output: Path) -> float:\\n    quality, quality_notes = code_quality(controller_path)\\n    score, feedback = score_runs(results, quality)\\n\\n    lines = [\\n        \\"VIRTUAL PICAR-X EVALUATION\\",\\n        \\"=\\" * 32,\\n        f\\"Automated score: {score}/90\\",\\n        f\\"Driving behavior: {round(score - quality, 1)}/80\\",\\n        f\\"Code quality: {quality}/10\\",\\n        \\"Instructor demonstration: ___/10\\",\\n        f\\"Possible final score: {score}/100 before demonstration points\\",\\n        \\"\\",\\n        \\"SCENARIOS\\",\\n    ]\\n    for result in results:\\n        lines.append(\\n            f\\"- {result.scenario:12} goal={str(result.reached_goal):5} \\"\\n            f\\"checkpoints={result.checkpoints_reached}/{result.total_checkpoints} \\"\\n            f\\"collisions={result.collisions} human_hits={result.human_hits} \\"\\n            f\\"stop_violations={result.stop_sign_violations} steps={result.steps}\\"\\n        )\\n        if result.replay and result.replay[-1][\\"failure\\"]:\\n            lines.append(f\\"  Attempt ended: {result.replay[-1][\'failure\']}\\")\\n        if result.controller_errors:\\n            lines.append(f\\"  Controller error: {result.controller_errors[0]}\\")\\n    lines.extend([\\"\\", \\"NEXT IMPROVEMENTS\\"])\\n    for note in feedback + quality_notes:\\n        lines.append(f\\"- {note}\\")\\n    if not feedback and not quality_notes:\\n        lines.append(\\"- Strong result. Explain why the design works and identify one remaining limitation.\\")\\n    report = \\"\\\\n\\".join(lines) + \\"\\\\n\\"\\n    output.write_text(report, encoding=\\"utf-8\\")\\n    print(report)\\n    return score\\n\\n\\ndef main() -> None:\\n    controller = load_controller(CONTROLLER)\\n    scenario = scenario_catalog()[\\"warehouse\\"]\\n    result = run_scenario(controller, scenario, strict=True)\\n    write_score_report([result], CONTROLLER, HERE / \\"score_report.txt\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "car_api.py": "\\"\\"\\"Course helpers shared by the virtual and physical SunFounder PiCar-X.\\"\\"\\"\\n\\nfrom collections.abc import Callable\\nfrom typing import Literal\\n\\nfrom vendor.sunfounder_picarx import Picarx\\n\\nScanLabel = Literal[\\"human\\", \\"stop sign\\", \\"clear\\"]\\n\\n\\nclass CoursePicarx(Picarx):\\n    \\"\\"\\"The SunFounder car API plus the course\'s camera and turning helpers.\\"\\"\\"\\n\\n    def scan(self) -> ScanLabel:\\n        \\"\\"\\"Read a camera label supplied by the selected environment.\\"\\"\\"\\n        raise NotImplementedError(\\"A camera scanner must supply this reading.\\")\\n\\n    def turn_left(self, speed: float = 28) -> None:\\n        self.set_dir_servo_angle(-28)\\n        self.forward(speed)\\n\\n    def turn_right(self, speed: float = 28) -> None:\\n        self.set_dir_servo_angle(28)\\n        self.forward(speed)\\n\\n\\nclass HardwarePicarx(CoursePicarx):\\n    \\"\\"\\"Use SunFounder\'s motor/sensor implementation and an injected scanner.\\"\\"\\"\\n\\n    is_simulation = False\\n\\n    def __init__(self, scanner: Callable[[], ScanLabel], **hardware_options) -> None:\\n        self._scanner = scanner\\n        super().__init__(**hardware_options)\\n\\n    def scan(self) -> ScanLabel:\\n        # Keep the motors stopped while a synchronous camera inference runs.\\n        self.stop()\\n        label = self._scanner()\\n        if label not in (\\"human\\", \\"stop sign\\", \\"clear\\"):\\n            self.stop()\\n            raise ValueError(f\\"Camera scanner returned an unsupported label: {label!r}\\")\\n        return label\\n", "vendor/__init__.py": "\\"\\"\\"Pinned upstream sources used by the PiCar-X course.\\"\\"\\"\\n", "vendor/sunfounder_picarx.py": "# SunFounder picar-x, revision 897698daf2719d830a5fb0b83951704bfee07d0c.\\n# Local change: allow the class to be imported on computers without robot_hat.\\n# Virtual subclasses supply their own initialization and sensor/motor methods.\\ntry:\\n    from robot_hat import Pin, ADC, PWM, Servo, fileDB\\n    from robot_hat import Grayscale_Module, Ultrasonic, utils\\nexcept ModuleNotFoundError as error:\\n    if error.name != \\"robot_hat\\":\\n        raise\\n    class _HardwareUnavailable:\\n        def __getattr__(self, name):\\n            raise RuntimeError(\\n                \\"Physical PiCar-X needs SunFounder\'s robot_hat on a Raspberry Pi.\\"\\n            )\\n    utils = _HardwareUnavailable()\\nimport time\\nimport os\\n\\n\\ndef constrain(x, min_val, max_val):\\n    \'\'\'\\n    Constrains value to be within a range.\\n    \'\'\'\\n    return max(min_val, min(max_val, x))\\n\\nclass Picarx(object):\\n    CONFIG = \'/opt/picar-x/picar-x.conf\'\\n\\n    DEFAULT_LINE_REF = [1000, 1000, 1000]\\n    DEFAULT_CLIFF_REF = [500, 500, 500]\\n\\n    DIR_MIN = -30\\n    DIR_MAX = 30\\n    CAM_PAN_MIN = -90\\n    CAM_PAN_MAX = 90\\n    CAM_TILT_MIN = -35\\n    CAM_TILT_MAX = 65\\n\\n    PERIOD = 4095\\n    PRESCALER = 10\\n    TIMEOUT = 0.02\\n\\n    # servo_pins: camera_pan_servo, camera_tilt_servo, direction_servo\\n    # motor_pins: left_swicth, right_swicth, left_pwm, right_pwm\\n    # grayscale_pins: 3 adc channels\\n    # ultrasonic_pins: trig, echo2\\n    # config: path of config file\\n    def __init__(self, \\n                servo_pins:list=[\'P0\', \'P1\', \'P2\'], \\n                motor_pins:list=[\'D4\', \'D5\', \'P13\', \'P12\'],\\n                grayscale_pins:list=[\'A0\', \'A1\', \'A2\'],\\n                ultrasonic_pins:list=[\'D2\',\'D3\'],\\n                config:str=CONFIG,\\n                ):\\n\\n        # reset robot_hat\\n        utils.reset_mcu()\\n        time.sleep(0.2)\\n\\n        # --------- config_flie ---------\\n        self.config_flie = fileDB(config, 777, os.getlogin())\\n\\n        # --------- servos init ---------\\n        self.cam_pan = Servo(servo_pins[0])\\n        self.cam_tilt = Servo(servo_pins[1])   \\n        self.dir_servo_pin = Servo(servo_pins[2])\\n        # get calibration values\\n        self.dir_cali_val = float(self.config_flie.get(\\"picarx_dir_servo\\", default_value=0))\\n        self.cam_pan_cali_val = float(self.config_flie.get(\\"picarx_cam_pan_servo\\", default_value=0))\\n        self.cam_tilt_cali_val = float(self.config_flie.get(\\"picarx_cam_tilt_servo\\", default_value=0))\\n        # set servos to init angle\\n        self.dir_servo_pin.angle(self.dir_cali_val)\\n        self.cam_pan.angle(self.cam_pan_cali_val)\\n        self.cam_tilt.angle(self.cam_tilt_cali_val)\\n\\n        # --------- motors init ---------\\n        self.left_rear_dir_pin = Pin(motor_pins[0])\\n        self.right_rear_dir_pin = Pin(motor_pins[1])\\n        self.left_rear_pwm_pin = PWM(motor_pins[2])\\n        self.right_rear_pwm_pin = PWM(motor_pins[3])\\n        self.motor_direction_pins = [self.left_rear_dir_pin, self.right_rear_dir_pin]\\n        self.motor_speed_pins = [self.left_rear_pwm_pin, self.right_rear_pwm_pin]\\n        # get calibration values\\n        self.cali_dir_value = self.config_flie.get(\\"picarx_dir_motor\\", default_value=\\"[1, 1]\\")\\n        self.cali_dir_value = [int(i.strip()) for i in self.cali_dir_value.strip().strip(\\"[]\\").split(\\",\\")]\\n        self.cali_speed_value = [0, 0]\\n        self.dir_current_angle = 0\\n        # init pwm\\n        for pin in self.motor_speed_pins:\\n            pin.period(self.PERIOD)\\n            pin.prescaler(self.PRESCALER)\\n\\n        # --------- grayscale module init ---------\\n        adc0, adc1, adc2 = [ADC(pin) for pin in grayscale_pins]\\n        self.grayscale = Grayscale_Module(adc0, adc1, adc2, reference=None)\\n        # get reference\\n        self.line_reference = self.config_flie.get(\\"line_reference\\", default_value=str(self.DEFAULT_LINE_REF))\\n        self.line_reference = [float(i) for i in self.line_reference.strip().strip(\'[]\').split(\',\')]\\n        self.cliff_reference = self.config_flie.get(\\"cliff_reference\\", default_value=str(self.DEFAULT_CLIFF_REF))\\n        self.cliff_reference = [float(i) for i in self.cliff_reference.strip().strip(\'[]\').split(\',\')]\\n        # transfer reference\\n        self.grayscale.reference(self.line_reference)\\n\\n        # --------- ultrasonic init ---------\\n        trig, echo= ultrasonic_pins\\n        self.ultrasonic = Ultrasonic(Pin(trig), Pin(echo, mode=Pin.IN, pull=Pin.PULL_DOWN))\\n        \\n    def set_motor_speed(self, motor, speed):\\n        \'\'\' set motor speed\\n        \\n        param motor: motor index, 1 means left motor, 2 means right motor\\n        type motor: int\\n        param speed: speed\\n        type speed: int      \\n        \'\'\'\\n        speed = constrain(speed, -100, 100)\\n        motor -= 1\\n        if speed >= 0:\\n            direction = 1 * self.cali_dir_value[motor]\\n        elif speed < 0:\\n            direction = -1 * self.cali_dir_value[motor]\\n        speed = abs(speed)\\n        # print(f\\"direction: {direction}, speed: {speed}\\")\\n        if speed != 0:\\n            speed = int(speed /2 ) + 50\\n        speed = speed - self.cali_speed_value[motor]\\n        if direction < 0:\\n            self.motor_direction_pins[motor].high()\\n            self.motor_speed_pins[motor].pulse_width_percent(speed)\\n        else:\\n            self.motor_direction_pins[motor].low()\\n            self.motor_speed_pins[motor].pulse_width_percent(speed)\\n\\n    def motor_speed_calibration(self, value):\\n        self.cali_speed_value = value\\n        if value < 0:\\n            self.cali_speed_value[0] = 0\\n            self.cali_speed_value[1] = abs(self.cali_speed_value)\\n        else:\\n            self.cali_speed_value[0] = abs(self.cali_speed_value)\\n            self.cali_speed_value[1] = 0\\n\\n    def motor_direction_calibrate(self, motor, value):\\n        \'\'\' set motor direction calibration value\\n        \\n        param motor: motor index, 1 means left motor, 2 means right motor\\n        type motor: int\\n        param value: speed\\n        type value: int\\n        \'\'\'      \\n        motor -= 1\\n        if value == 1:\\n            self.cali_dir_value[motor] = 1\\n        elif value == -1:\\n            self.cali_dir_value[motor] = -1\\n        self.config_flie.set(\\"picarx_dir_motor\\", self.cali_dir_value)\\n\\n    def dir_servo_calibrate(self, value):\\n        self.dir_cali_val = value\\n        self.config_flie.set(\\"picarx_dir_servo\\", \\"%s\\"%value)\\n        self.dir_servo_pin.angle(value)\\n\\n    def set_dir_servo_angle(self, value):\\n        self.dir_current_angle = constrain(value, self.DIR_MIN, self.DIR_MAX)\\n        angle_value  = self.dir_current_angle + self.dir_cali_val\\n        self.dir_servo_pin.angle(angle_value)\\n\\n    def cam_pan_servo_calibrate(self, value):\\n        self.cam_pan_cali_val = value\\n        self.config_flie.set(\\"picarx_cam_pan_servo\\", \\"%s\\"%value)\\n        self.cam_pan.angle(value)\\n\\n    def cam_tilt_servo_calibrate(self, value):\\n        self.cam_tilt_cali_val = value\\n        self.config_flie.set(\\"picarx_cam_tilt_servo\\", \\"%s\\"%value)\\n        self.cam_tilt.angle(value)\\n\\n    def set_cam_pan_angle(self, value):\\n        value = constrain(value, self.CAM_PAN_MIN, self.CAM_PAN_MAX)\\n        self.cam_pan.angle(-1*(value + -1*self.cam_pan_cali_val))\\n\\n    def set_cam_tilt_angle(self,value):\\n        value = constrain(value, self.CAM_TILT_MIN, self.CAM_TILT_MAX)\\n        self.cam_tilt.angle(-1*(value + -1*self.cam_tilt_cali_val))\\n\\n    def set_power(self, speed):\\n        self.set_motor_speed(1, speed)\\n        self.set_motor_speed(2, speed)\\n\\n    def backward(self, speed):\\n        current_angle = self.dir_current_angle\\n        if current_angle != 0:\\n            abs_current_angle = abs(current_angle)\\n            if abs_current_angle > self.DIR_MAX:\\n                abs_current_angle = self.DIR_MAX\\n            power_scale = (100 - abs_current_angle) / 100.0 \\n            if (current_angle / abs_current_angle) > 0:\\n                self.set_motor_speed(1, -1*speed)\\n                self.set_motor_speed(2, speed * power_scale)\\n            else:\\n                self.set_motor_speed(1, -1*speed * power_scale)\\n                self.set_motor_speed(2, speed )\\n        else:\\n            self.set_motor_speed(1, -1*speed)\\n            self.set_motor_speed(2, speed)  \\n\\n    def forward(self, speed):\\n        current_angle = self.dir_current_angle\\n        if current_angle != 0:\\n            abs_current_angle = abs(current_angle)\\n            if abs_current_angle > self.DIR_MAX:\\n                abs_current_angle = self.DIR_MAX\\n            power_scale = (100 - abs_current_angle) / 100.0\\n            if (current_angle / abs_current_angle) > 0:\\n                self.set_motor_speed(1, 1*speed * power_scale)\\n                self.set_motor_speed(2, -speed) \\n            else:\\n                self.set_motor_speed(1, speed)\\n                self.set_motor_speed(2, -1*speed * power_scale)\\n        else:\\n            self.set_motor_speed(1, speed)\\n            self.set_motor_speed(2, -1*speed)                  \\n\\n    def stop(self):\\n        \'\'\'\\n        Execute twice to make sure it stops\\n        \'\'\'\\n        for _ in range(2):\\n            self.motor_speed_pins[0].pulse_width_percent(0)\\n            self.motor_speed_pins[1].pulse_width_percent(0)\\n            time.sleep(0.002)\\n\\n    def get_distance(self):\\n        return self.ultrasonic.read()\\n\\n    def set_grayscale_reference(self, value):\\n        if isinstance(value, list) and len(value) == 3:\\n            self.line_reference = value\\n            self.grayscale.reference(self.line_reference)\\n            self.config_flie.set(\\"line_reference\\", self.line_reference)\\n        else:\\n            raise ValueError(\\"grayscale reference must be a 1*3 list\\")\\n\\n    def get_grayscale_data(self):\\n        return list.copy(self.grayscale.read())\\n\\n    def get_line_status(self,gm_val_list):\\n        return self.grayscale.read_status(gm_val_list)\\n\\n    def set_line_reference(self, value):\\n        self.set_grayscale_reference(value)\\n\\n    def get_cliff_status(self,gm_val_list):\\n        for i in range(0,3):\\n            if gm_val_list[i]<=self.cliff_reference[i]:\\n                return True\\n        return False\\n\\n    def set_cliff_reference(self, value):\\n        if isinstance(value, list) and len(value) == 3:\\n            self.cliff_reference = value\\n            self.config_flie.set(\\"cliff_reference\\", self.cliff_reference)\\n        else:\\n            raise ValueError(\\"grayscale reference must be a 1*3 list\\")\\n\\n    def reset(self):\\n        self.stop()\\n        self.set_dir_servo_angle(0)\\n        self.set_cam_tilt_angle(0)\\n        self.set_cam_pan_angle(0)\\n\\n    def close(self):\\n        self.reset()\\n        self.ultrasonic.close()\\n\\nif __name__ == \\"__main__\\":\\n    px = Picarx()\\n    px.forward(50)\\n    time.sleep(1)\\n    px.stop()\\n", "vendor/LICENSE.picar-x": "                    GNU GENERAL PUBLIC LICENSE\\n                       Version 2, June 1991\\n\\n Copyright (C) 1989, 1991 Free Software Foundation, Inc., <http://fsf.org/>\\n 51 Franklin Street, Fifth Floor, Boston, MA 02110-1301 USA\\n Everyone is permitted to copy and distribute verbatim copies\\n of this license document, but changing it is not allowed.\\n\\n                            Preamble\\n\\n  The licenses for most software are designed to take away your\\nfreedom to share and change it.  By contrast, the GNU General Public\\nLicense is intended to guarantee your freedom to share and change free\\nsoftware--to make sure the software is free for all its users.  This\\nGeneral Public License applies to most of the Free Software\\nFoundation\'s software and to any other program whose authors commit to\\nusing it.  (Some other Free Software Foundation software is covered by\\nthe GNU Lesser General Public License instead.)  You can apply it to\\nyour programs, too.\\n\\n  When we speak of free software, we are referring to freedom, not\\nprice.  Our General Public Licenses are designed to make sure that you\\nhave the freedom to distribute copies of free software (and charge for\\nthis service if you wish), that you receive source code or can get it\\nif you want it, that you can change the software or use pieces of it\\nin new free programs; and that you know you can do these things.\\n\\n  To protect your rights, we need to make restrictions that forbid\\nanyone to deny you these rights or to ask you to surrender the rights.\\nThese restrictions translate to certain responsibilities for you if you\\ndistribute copies of the software, or if you modify it.\\n\\n  For example, if you distribute copies of such a program, whether\\ngratis or for a fee, you must give the recipients all the rights that\\nyou have.  You must make sure that they, too, receive or can get the\\nsource code.  And you must show them these terms so they know their\\nrights.\\n\\n  We protect your rights with two steps: (1) copyright the software, and\\n(2) offer you this license which gives you legal permission to copy,\\ndistribute and/or modify the software.\\n\\n  Also, for each author\'s protection and ours, we want to make certain\\nthat everyone understands that there is no warranty for this free\\nsoftware.  If the software is modified by someone else and passed on, we\\nwant its recipients to know that what they have is not the original, so\\nthat any problems introduced by others will not reflect on the original\\nauthors\' reputations.\\n\\n  Finally, any free program is threatened constantly by software\\npatents.  We wish to avoid the danger that redistributors of a free\\nprogram will individually obtain patent licenses, in effect making the\\nprogram proprietary.  To prevent this, we have made it clear that any\\npatent must be licensed for everyone\'s free use or not licensed at all.\\n\\n  The precise terms and conditions for copying, distribution and\\nmodification follow.\\n\\n                    GNU GENERAL PUBLIC LICENSE\\n   TERMS AND CONDITIONS FOR COPYING, DISTRIBUTION AND MODIFICATION\\n\\n  0. This License applies to any program or other work which contains\\na notice placed by the copyright holder saying it may be distributed\\nunder the terms of this General Public License.  The \\"Program\\", below,\\nrefers to any such program or work, and a \\"work based on the Program\\"\\nmeans either the Program or any derivative work under copyright law:\\nthat is to say, a work containing the Program or a portion of it,\\neither verbatim or with modifications and/or translated into another\\nlanguage.  (Hereinafter, translation is included without limitation in\\nthe term \\"modification\\".)  Each licensee is addressed as \\"you\\".\\n\\nActivities other than copying, distribution and modification are not\\ncovered by this License; they are outside its scope.  The act of\\nrunning the Program is not restricted, and the output from the Program\\nis covered only if its contents constitute a work based on the\\nProgram (independent of having been made by running the Program).\\nWhether that is true depends on what the Program does.\\n\\n  1. You may copy and distribute verbatim copies of the Program\'s\\nsource code as you receive it, in any medium, provided that you\\nconspicuously and appropriately publish on each copy an appropriate\\ncopyright notice and disclaimer of warranty; keep intact all the\\nnotices that refer to this License and to the absence of any warranty;\\nand give any other recipients of the Program a copy of this License\\nalong with the Program.\\n\\nYou may charge a fee for the physical act of transferring a copy, and\\nyou may at your option offer warranty protection in exchange for a fee.\\n\\n  2. You may modify your copy or copies of the Program or any portion\\nof it, thus forming a work based on the Program, and copy and\\ndistribute such modifications or work under the terms of Section 1\\nabove, provided that you also meet all of these conditions:\\n\\n    a) You must cause the modified files to carry prominent notices\\n    stating that you changed the files and the date of any change.\\n\\n    b) You must cause any work that you distribute or publish, that in\\n    whole or in part contains or is derived from the Program or any\\n    part thereof, to be licensed as a whole at no charge to all third\\n    parties under the terms of this License.\\n\\n    c) If the modified program normally reads commands interactively\\n    when run, you must cause it, when started running for such\\n    interactive use in the most ordinary way, to print or display an\\n    announcement including an appropriate copyright notice and a\\n    notice that there is no warranty (or else, saying that you provide\\n    a warranty) and that users may redistribute the program under\\n    these conditions, and telling the user how to view a copy of this\\n    License.  (Exception: if the Program itself is interactive but\\n    does not normally print such an announcement, your work based on\\n    the Program is not required to print an announcement.)\\n\\nThese requirements apply to the modified work as a whole.  If\\nidentifiable sections of that work are not derived from the Program,\\nand can be reasonably considered independent and separate works in\\nthemselves, then this License, and its terms, do not apply to those\\nsections when you distribute them as separate works.  But when you\\ndistribute the same sections as part of a whole which is a work based\\non the Program, the distribution of the whole must be on the terms of\\nthis License, whose permissions for other licensees extend to the\\nentire whole, and thus to each and every part regardless of who wrote it.\\n\\nThus, it is not the intent of this section to claim rights or contest\\nyour rights to work written entirely by you; rather, the intent is to\\nexercise the right to control the distribution of derivative or\\ncollective works based on the Program.\\n\\nIn addition, mere aggregation of another work not based on the Program\\nwith the Program (or with a work based on the Program) on a volume of\\na storage or distribution medium does not bring the other work under\\nthe scope of this License.\\n\\n  3. You may copy and distribute the Program (or a work based on it,\\nunder Section 2) in object code or executable form under the terms of\\nSections 1 and 2 above provided that you also do one of the following:\\n\\n    a) Accompany it with the complete corresponding machine-readable\\n    source code, which must be distributed under the terms of Sections\\n    1 and 2 above on a medium customarily used for software interchange; or,\\n\\n    b) Accompany it with a written offer, valid for at least three\\n    years, to give any third party, for a charge no more than your\\n    cost of physically performing source distribution, a complete\\n    machine-readable copy of the corresponding source code, to be\\n    distributed under the terms of Sections 1 and 2 above on a medium\\n    customarily used for software interchange; or,\\n\\n    c) Accompany it with the information you received as to the offer\\n    to distribute corresponding source code.  (This alternative is\\n    allowed only for noncommercial distribution and only if you\\n    received the program in object code or executable form with such\\n    an offer, in accord with Subsection b above.)\\n\\nThe source code for a work means the preferred form of the work for\\nmaking modifications to it.  For an executable work, complete source\\ncode means all the source code for all modules it contains, plus any\\nassociated interface definition files, plus the scripts used to\\ncontrol compilation and installation of the executable.  However, as a\\nspecial exception, the source code distributed need not include\\nanything that is normally distributed (in either source or binary\\nform) with the major components (compiler, kernel, and so on) of the\\noperating system on which the executable runs, unless that component\\nitself accompanies the executable.\\n\\nIf distribution of executable or object code is made by offering\\naccess to copy from a designated place, then offering equivalent\\naccess to copy the source code from the same place counts as\\ndistribution of the source code, even though third parties are not\\ncompelled to copy the source along with the object code.\\n\\n  4. You may not copy, modify, sublicense, or distribute the Program\\nexcept as expressly provided under this License.  Any attempt\\notherwise to copy, modify, sublicense or distribute the Program is\\nvoid, and will automatically terminate your rights under this License.\\nHowever, parties who have received copies, or rights, from you under\\nthis License will not have their licenses terminated so long as such\\nparties remain in full compliance.\\n\\n  5. You are not required to accept this License, since you have not\\nsigned it.  However, nothing else grants you permission to modify or\\ndistribute the Program or its derivative works.  These actions are\\nprohibited by law if you do not accept this License.  Therefore, by\\nmodifying or distributing the Program (or any work based on the\\nProgram), you indicate your acceptance of this License to do so, and\\nall its terms and conditions for copying, distributing or modifying\\nthe Program or works based on it.\\n\\n  6. Each time you redistribute the Program (or any work based on the\\nProgram), the recipient automatically receives a license from the\\noriginal licensor to copy, distribute or modify the Program subject to\\nthese terms and conditions.  You may not impose any further\\nrestrictions on the recipients\' exercise of the rights granted herein.\\nYou are not responsible for enforcing compliance by third parties to\\nthis License.\\n\\n  7. If, as a consequence of a court judgment or allegation of patent\\ninfringement or for any other reason (not limited to patent issues),\\nconditions are imposed on you (whether by court order, agreement or\\notherwise) that contradict the conditions of this License, they do not\\nexcuse you from the conditions of this License.  If you cannot\\ndistribute so as to satisfy simultaneously your obligations under this\\nLicense and any other pertinent obligations, then as a consequence you\\nmay not distribute the Program at all.  For example, if a patent\\nlicense would not permit royalty-free redistribution of the Program by\\nall those who receive copies directly or indirectly through you, then\\nthe only way you could satisfy both it and this License would be to\\nrefrain entirely from distribution of the Program.\\n\\nIf any portion of this section is held invalid or unenforceable under\\nany particular circumstance, the balance of the section is intended to\\napply and the section as a whole is intended to apply in other\\ncircumstances.\\n\\nIt is not the purpose of this section to induce you to infringe any\\npatents or other property right claims or to contest validity of any\\nsuch claims; this section has the sole purpose of protecting the\\nintegrity of the free software distribution system, which is\\nimplemented by public license practices.  Many people have made\\ngenerous contributions to the wide range of software distributed\\nthrough that system in reliance on consistent application of that\\nsystem; it is up to the author/donor to decide if he or she is willing\\nto distribute software through any other system and a licensee cannot\\nimpose that choice.\\n\\nThis section is intended to make thoroughly clear what is believed to\\nbe a consequence of the rest of this License.\\n\\n  8. If the distribution and/or use of the Program is restricted in\\ncertain countries either by patents or by copyrighted interfaces, the\\noriginal copyright holder who places the Program under this License\\nmay add an explicit geographical distribution limitation excluding\\nthose countries, so that distribution is permitted only in or among\\ncountries not thus excluded.  In such case, this License incorporates\\nthe limitation as if written in the body of this License.\\n\\n  9. The Free Software Foundation may publish revised and/or new versions\\nof the General Public License from time to time.  Such new versions will\\nbe similar in spirit to the present version, but may differ in detail to\\naddress new problems or concerns.\\n\\nEach version is given a distinguishing version number.  If the Program\\nspecifies a version number of this License which applies to it and \\"any\\nlater version\\", you have the option of following the terms and conditions\\neither of that version or of any later version published by the Free\\nSoftware Foundation.  If the Program does not specify a version number of\\nthis License, you may choose any version ever published by the Free Software\\nFoundation.\\n\\n  10. If you wish to incorporate parts of the Program into other free\\nprograms whose distribution conditions are different, write to the author\\nto ask for permission.  For software which is copyrighted by the Free\\nSoftware Foundation, write to the Free Software Foundation; we sometimes\\nmake exceptions for this.  Our decision will be guided by the two goals\\nof preserving the free status of all derivatives of our free software and\\nof promoting the sharing and reuse of software generally.\\n\\n                            NO WARRANTY\\n\\n  11. BECAUSE THE PROGRAM IS LICENSED FREE OF CHARGE, THERE IS NO WARRANTY\\nFOR THE PROGRAM, TO THE EXTENT PERMITTED BY APPLICABLE LAW.  EXCEPT WHEN\\nOTHERWISE STATED IN WRITING THE COPYRIGHT HOLDERS AND/OR OTHER PARTIES\\nPROVIDE THE PROGRAM \\"AS IS\\" WITHOUT WARRANTY OF ANY KIND, EITHER EXPRESSED\\nOR IMPLIED, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF\\nMERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE.  THE ENTIRE RISK AS\\nTO THE QUALITY AND PERFORMANCE OF THE PROGRAM IS WITH YOU.  SHOULD THE\\nPROGRAM PROVE DEFECTIVE, YOU ASSUME THE COST OF ALL NECESSARY SERVICING,\\nREPAIR OR CORRECTION.\\n\\n  12. IN NO EVENT UNLESS REQUIRED BY APPLICABLE LAW OR AGREED TO IN WRITING\\nWILL ANY COPYRIGHT HOLDER, OR ANY OTHER PARTY WHO MAY MODIFY AND/OR\\nREDISTRIBUTE THE PROGRAM AS PERMITTED ABOVE, BE LIABLE TO YOU FOR DAMAGES,\\nINCLUDING ANY GENERAL, SPECIAL, INCIDENTAL OR CONSEQUENTIAL DAMAGES ARISING\\nOUT OF THE USE OR INABILITY TO USE THE PROGRAM (INCLUDING BUT NOT LIMITED\\nTO LOSS OF DATA OR DATA BEING RENDERED INACCURATE OR LOSSES SUSTAINED BY\\nYOU OR THIRD PARTIES OR A FAILURE OF THE PROGRAM TO OPERATE WITH ANY OTHER\\nPROGRAMS), EVEN IF SUCH HOLDER OR OTHER PARTY HAS BEEN ADVISED OF THE\\nPOSSIBILITY OF SUCH DAMAGES.\\n\\n                     END OF TERMS AND CONDITIONS\\n", "guided_course.py": "\\"\\"\\"Continuous, fixed-world skills course behind the beginner camera interface.\\n\\nThe camera is mocked. The custom detour uses a guided lane-change curve,\\nnot a vehicle dynamics model. Students only supply sensor/action decisions.\\n\\"\\"\\"\\nimport math\\n\\n\\ndef run_guided_course(controller, obstacle, direction, car_factory):\\n    from simulator import Scenario, Checkpoint, RunResult, SceneObject\\n\\n    if direction not in (\\"left\\", \\"right\\"):\\n        raise ValueError(\\"Choose left or right.\\")\\n    if not isinstance(obstacle, str) or not obstacle.strip() or obstacle in (\\"clear\\", \\"human\\", \\"stop sign\\", \\"unknown\\"):\\n        raise ValueError(\\"Choose a new, nonempty personal-object label.\\")\\n    car = car_factory()\\n    memory = {}\\n    frames = []\\n    x, y, heading = 65.0, 260.0, 0.0\\n    traveled = 0.0\\n    sign_wait = 0\\n    sign_done = False\\n    crossing_start = None\\n    box_wait = 0\\n    box_move_start = None\\n    failure = \\"\\"\\n    completed = 0\\n    changes = 0\\n    previous_angle = 0\\n    side = -1 if direction == \\"left\\" else 1\\n    checkpoint_x = (150, 340, 550, 850, 1450)\\n    scene = (\\n        SceneObject(\\"human\\", 300, 165, 12, (300, 355)),\\n        SceneObject(\\"stop sign\\", 480, 355, 18),\\n        SceneObject(\\"box\\", 760, 260, 18),\\n        SceneObject(obstacle, 1110, 260, 18),\\n        SceneObject(\\"human\\", 785, 390, 12),\\n    )\\n    for tick in range(2200):\\n        if x >= 175 and crossing_start is None:\\n            crossing_start = tick\\n        crossing = min(1.0, (tick - crossing_start) / 100) if crossing_start is not None else 0.0\\n        person_y = 165 + 190 * crossing\\n        box_progress = min(1.0, (tick-box_move_start)/70) if box_move_start is not None else 0.0\\n        box_y = 260 + 130 * box_progress\\n        worker_y = box_y + 35\\n        person_in_path = crossing_start is not None and crossing < 1 and 300-x < 105\\n        worker_in_path = box_move_start is not None and worker_y < 325\\n        custom_in_view = 960 <= x < 1260\\n        distance_cm = 200.0\\n        for bx, by in ((760, box_y), (1110, 260)):\\n            if bx > x and abs(by-y) < 30:\\n                distance_cm = min(distance_cm, max(0, bx-x-18))\\n        if person_in_path or worker_in_path:\\n            label = \\"human\\"\\n        elif 400 <= x < 500 and not sign_done:\\n            label = \\"stop sign\\"\\n        elif custom_in_view:\\n            label = obstacle\\n        else:\\n            label = \\"clear\\"\\n        car.label, car.distance, car.history = label, distance_cm, []\\n        try:\\n            controller(car, memory)\\n        except Exception as error:\\n            failure = f\\"Controller error: {type(error).__name__}: {error}\\"\\n        speed = car.history[-1][1] if car.history and car.action == \\"forward\\" else 0\\n        moving = speed > 0\\n        issued_forward = any(action == \\"forward\\" for action, power in car.history)\\n        must_stop = label in (\\"human\\", \\"stop sign\\") or distance_cm < 25\\n        if not failure and must_stop and (moving or issued_forward):\\n            failure = (\\"Yield to the pedestrian\\" if label == \\"human\\" else\\n                       \\"Stop at the sign\\" if label == \\"stop sign\\" else\\n                       \\"Stop before the nearby box\\")\\n        if not failure and moving and label == obstacle and car.steering != side*25:\\n            failure = f\\"Steer {direction} around {obstacle}: use angle {side*25}.\\"\\n        if not failure and moving and label == \\"clear\\" and car.steering != 0:\\n            failure = \\"Straighten the wheels when the view clears.\\"\\n        if label == \\"stop sign\\" and not moving and not failure:\\n            sign_wait += 1\\n            sign_done = sign_wait >= 12\\n        if 690 <= x < 760 and distance_cm < 25 and not moving and not failure:\\n            box_wait += 1\\n            if box_wait >= 8 and box_move_start is None:\\n                box_move_start = tick\\n        if not failure and moving:\\n            previous_x, previous_y = x, y\\n            x += speed * 0.72 * 0.12\\n            # The course adapter guides the lane change; no path math in student code.\\n            t = max(0.0, min(1.0, (x-960)/300))\\n            y = 260 + side * 70 * math.sin(math.pi*t)**2\\n            heading = math.degrees(math.atan(side*70*math.pi/300*math.sin(2*math.pi*t)))\\n            traveled += math.hypot(x-previous_x, y-previous_y)\\n            if x >= 455 and not sign_done:\\n                failure = \\"Ran the stop sign\\"\\n            for bx, by in ((760, box_y), (1110,260)):\\n                if math.hypot(x-bx,y-by) < 30:\\n                    failure = \\"Hit an obstacle\\"\\n            if math.hypot(x-300,y-person_y) < 24:\\n                failure = \\"Hit a pedestrian\\"\\n        completed = sum(x >= point for point in checkpoint_x)\\n        objects = [\\n            dict(id=\\"crossing-person\\",kind=\\"human\\",x=300,y=person_y,radius=12,walking=0<crossing<1),\\n            dict(id=\\"roadside-sign\\",kind=\\"stop sign\\",x=480,y=355,radius=18,walking=False),\\n            dict(id=\\"movable-box\\",kind=\\"box\\",x=760,y=box_y,radius=18,walking=False),\\n            dict(id=\\"personal-object\\",kind=obstacle,x=1110,y=260,radius=18,walking=False),\\n            dict(id=\\"worker\\",kind=\\"human\\",x=785,y=worker_y,radius=12,walking=0<box_progress<1),\\n        ]\\n        changes += car.steering != previous_angle\\n        previous_angle = car.steering\\n        frames.append(dict(step=tick,x=round(x,2),y=round(y,2),heading=round(heading,2),\\n            steering=car.steering,throttle=0 if failure else speed,checkpoint=completed,\\n            scan=label,objects=objects,failure=failure))\\n        if completed == 5 or failure:\\n            break\\n        if len(frames)>240 and math.hypot(x-frames[-240][\\"x\\"], y-frames[-240][\\"y\\"])<2:\\n            failure = \\"No progress: check for an unconditional stop or an early return.\\"\\n            frames[-1][\\"failure\\"] = failure\\n            break\\n    else:\\n        failure = \\"Course time limit reached. Use forward(30) when the road is clear.\\"\\n        frames[-1][\\"failure\\"] = failure\\n    success = completed == 5 and not failure\\n    frames[-1][\\"throttle\\"] = 0\\n    scenario = Scenario(\\"guided_neighborhood\\",1600,520,(65,260,0),Checkpoint(1450,260,32),(),\\n        tuple(Checkpoint(cx,260,32) for cx in checkpoint_x),((40,260),(1550,260)),scene,\\n        max_steps=2200,road_half_width=85)\\n    result = RunResult(scenario=scenario.name,reached_goal=success,checkpoints_reached=completed,\\n        total_checkpoints=5,collisions=int(failure==\\"Hit an obstacle\\"),boundary_hits=0,\\n        invalid_commands=0,sensor_faults_seen=0,steps=len(frames),distance_traveled=round(traveled,2),\\n        steering_changes=changes,stopped_safely=True,human_hits=int(failure==\\"Hit a pedestrian\\"),\\n        stop_sign_violations=int(failure==\\"Ran the stop sign\\"),stop_signs_obeyed=int(sign_done),replay=frames)\\n    print(f\\"Skills checkpoints: {completed}/5\\")\\n    if failure:\\n        print(\\"TRY AGAIN:\\",failure)\\n    return result, scenario\\n"}')
for name, contents in bundled_sources.items():
    destination = work_folder / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents, encoding="utf-8")
sys.path.insert(0, str(work_folder))
print("Ready! Edit and run your controller cell below.")


from guided_course import run_guided_course as _continuous_course

from guided_course import run_guided_course as _continuous_course

def run_guided_course(controller, obstacle, direction):
    return _continuous_course(controller, obstacle, direction, MockPicarx)


## 2. Working controller — run as supplied


In [ ]:
driver_name = "Captain Cautious"
custom_object = "giant sneaker"
avoid_direction = "left"


def control_step(car, memory: dict) -> None:
    """Read both sensors, handle hazards, then move when allowed."""
    scan_input = car.scan()
    distance = car.ultrasonic.read()

    if scan_input == "human":
        car.stop()
        return

    if scan_input == "stop sign":
        car.stop()
        return

    if distance < 25:
        car.stop()
        return

    if scan_input == custom_object:
        if avoid_direction == "left":
            car.set_dir_servo_angle(-25)
        else:
            car.set_dir_servo_angle(25)
        car.forward(30)
        return

    if scan_input != "clear":
        car.stop()
        return

    car.set_dir_servo_angle(0)
    car.forward(30)


my_test = [
    ("clear", 100),
    ("human", 70),
    ("stop sign", 70),
    (custom_object, 70),
    (custom_object, 10),
    ("clear", 100),
]

print("Working guided controller loaded. Run the replay cell next.")


## 3. Simulation — run, then watch below


In [ ]:
from html import escape
from IPython.display import HTML, display
from simulator import render_replays

result, scenario = run_guided_course(control_step, custom_object, avoid_direction)
if not result.reached_goal:
    print("Course stopped:", result.replay[-1]["failure"])
else:
    print("COMPLETE: 5/5 skills checkpoints passed.")
replay_path = work_folder / "working_skills_replay.html"
render_replays([(result, scenario)], replay_path, autoplay=True)
page = replay_path.read_text(encoding="utf-8")
display(HTML('<iframe title="Working PiCar-X replay" style="width:100%;height:880px;border:0" '
    'sandbox="allow-scripts" srcdoc="' + escape(page, quote=True) + '"></iframe>'))
print("HTML replay saved:", replay_path)
